# Unitary Patent (`REG711_LICENSEE`)

The table `REG711_LICENSEE`contains information about licensees and rights *in rem* for Unitary Patents.
Let’s break down the key fields in this table. The first step as usual is to initialise the PATSTAT client and import the table.

In [2]:
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the models
from epo.tipdata.patstat.database.models import REG711_LICENSEE

# Importing sql alchemy package
from sqlalchemy import func

## Key fields in the `REG711_LICENSEE` table
* `ID` (primary key): a technical identifier for the Unitary Patent, without business meaning.
* `CHANGE_DATE`: this attribute indicates the date when the data was modified in the corresponding IT application of the EPO. It represents the date on which the record was saved in the database.
* `BULLETIN_YEAR`: this attribute indicates the year of the publication in the European Patent Bulletin.
* `BULLETIN_NR`: this attribute indicates indicates the calendar week in which the European Patent Bulletin has been published.
* `LICENSEE_SEQ_NR`: this atribute contins the serial number of license / sub-license. The first two digits identify the serial number of the main licence, while the optional additional two digits indicate the serial number of a sub-licence. Both main licences and sub-licences can have a maximum serial number of 10.
* `TYPE_LICENSE`: this attribute contains the type of license. There can be three types of license: exclusive, not exclusive or right *in rem*.
* `VALID_DATE`: this attribute contains the date when the license became valid.
* `NAME`: this attribute contains the names of natural and legal persons.
* `CUSTOMER_ID`: this attribute contains the identifiers of customers, who may be legal or natural persons. If a customer holds multiple roles (applicant, inventor, representative, licensee, opponent), they usually have separate CUSTOMER_IDs for each role.
* `ADDRESS_1`, `ADDRESS_2`,`ADDRESS_3`, `ADDRESS_4`, `ADDRESS_5`: these attributes contain the first / second / third / forth / fifth address line of a person. Address lines from the second onward are often empty.
* `COUNTRY`: this attribute contains the country/territory code. It is a two-letter country/territory code for patent parties (applicant/inventor/agent), designated states of applicant, or country of licensees.
* `DESIGNATION`: this attribute indicates the type of designation. 



Let's plot the table to visualise its structure.

In [3]:
reg711 = (
    db.query(
        REG711_LICENSEE.id,
        REG711_LICENSEE.bulletin_year,
        REG711_LICENSEE.bulletin_nr,
        REG711_LICENSEE.designation,
        REG711_LICENSEE.customer_id,
        REG711_LICENSEE.address_1,
        REG711_LICENSEE.country,
        REG711_LICENSEE.licensee_seq_nr,
        REG711_LICENSEE.type_license,
        REG711_LICENSEE.valid_date,
    )
    .order_by(REG711_LICENSEE.id)
    .limit(10000)
)

reg711_df = patstat.df(reg711)
reg711_df


,id,bulletin_year,bulletin_nr,designation,customer_id,address_1,country,licensee_seq_nr,type_license,valid_date
0,9770708,0,0,as-indicated,,,,03,RIR,2025-02-21
1,14788510,0,0,as-indicated,,,,03,RIR,2025-02-21
2,15733112,0,0,as-indicated,,,,01,RIR,2024-04-22
3,15808357,0,0,as-indicated,,,,01,RIR,2025-05-16
4,16767132,0,0,as-indicated,,,,05,RIR,2025-06-18
5,16767132,0,0,as-indicated,,,,04,RIR,2025-06-18
6,16767132,0,0,as-indicated,,,,03,RIR,2025-06-18
7,17720659,0,0,as-indicated,,,,05,RIR,2025-06-18
8,17720659,0,0,as-indicated,,,,04,RIR,2024-02-20
9,17720659,0,0,as-indicated,,,,03,RIR,2024-02-20


# Focus on the attributes `TYPE_LICENSE` and `LICENSEE_SEQ_NR`
As previously stated, the type of the license can assume three values: exclusive, not exclusive or right *in rem*. The domain consists of three ASCII characters:
* 'NOL', i.e. no license (occurs if and only if `licensee_seq_nr` = ‘deleted’)
* 'EXC', i.e. exclusive licence
* 'NEX', i.e. non-exclusive licence
* 'RIR', i.e. right *in rem*
It is possible to check the one-to-one relation between the values 'NOL' and 'deleted' from attributes `type_license` and `licensee_seq_nr` respectively.
Let's first count how many occurence of `licensee_seq_nr` have the value ‘deleted’

In [19]:
deleted = db.query(
    REG711_LICENSEE.licensee_seq_nr,
    func.count(REG711_LICENSEE.id).label('number of deleted')
).group_by(
    REG711_LICENSEE.licensee_seq_nr
).filter(
    REG711_LICENSEE.licensee_seq_nr == 'deleted'
)

deleted_df = patstat.df(deleted)
deleted_df

""


Note: since the output is empty, this means that there are no occurrences of  `licensee_seq_nr` with the value deleted in this table. Therefore, if NOL occurs only when `licensee_seq_nr` = 'deleted', we also expect 0 occurrences of 'NOL' for the `type_license` attribute.

In [20]:
Nol = db.query(
    REG711_LICENSEE.type_license,
    func.count(REG711_LICENSEE.id).label('number of NOL')
).group_by(
    REG711_LICENSEE.type_license
).filter(
    REG711_LICENSEE.type_license == 'NOL'
)

Nol_df = patstat.df(Nol)
Nol_df

""


# Focus on the attribute `VALID_DATE`
As mentioned before, the attribute `VALID_DATE` contains the date when the license became valid. It could be interesting to check how many licenses became valid after a certain date,
For instance, let's check how many licences became valid after '2012-09-02'.

In [21]:
from sqlalchemy import and_

valid_date = db.query(
    REG711_LICENSEE.id,
    REG711_LICENSEE.valid_date
).filter(
    and_(
        REG711_LICENSEE.valid_date > '2012-09-02',
        REG711_LICENSEE.valid_date != '9999-12-31'
    )
)

valid_date_df = patstat.df(valid_date)
valid_date_df


,id,valid_date
0,17787391,2023-10-25
1,21709141,2024-03-20
2,17771380,2024-03-19
3,20207552,2024-03-19
4,20834101,2024-03-19
5,17826641,2024-03-19
6,17826640,2024-03-19
7,18722771,2024-03-19
8,18830609,2024-03-19
9,17826643,2024-03-19
